In [6]:
%matplotlib inline

import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE" # tells OpenMP to not complain if it notices that two copies of OpenMP are loaded.

import sys
sys.path.append('../../../')


In [22]:
from torchcodec.encoders import VideoEncoder

from IPython.display import Video

def play_video(encoded_bytes):
    return Video(data=encoded_bytes.numpy().tobytes(),
                embed=True, width=640, height=360, mimetype="video/mp4")

In [50]:
import os
from pathlib import Path
from typing import Any, Callable, Optional, Union, cast

import torch
from torchcodec.decoders import VideoDecoder

%load_ext autoreload
%autoreload 2

from computer_vision.torch_video.data.dataset import VisionDataset
from computer_vision.torch_video.data.utils import find_classes, has_file_allowed_extension, make_dataset, compute_clip_start_times


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [33]:
# # https://meta-pytorch.org/torchcodec/stable/generated/torchcodec.samplers.clips_at_regular_timestamps.html#torchcodec.samplers.clips_at_regular_timestamps
# # https://meta-pytorch.org/torchcodec/stable/generated_examples/decoding/sampling.html
# # https://meta-pytorch.org/torchcodec/stable/generated_examples/index.html
# class UCF101(VisionDataset):
#     """`UCF101 <https://www.crcv.ucf.edu/data/UCF101.php>`_ dataset.
    
#     UCF101 is an action recognition video dataset. 
#     This dataset consider every video as a collection of video clips of fixed size, specified by ``frames_per_clip``, where the step in frames
#     between each clip is given by ``step_between_clips``. The dataset itself can be downloaded from the dataset website; annotations that 
#     ``annotation_path`` should be pointing to can be downloaded from `here <https://www.crcv.ucf.edu/data/UCF101/UCF101TrainTestSplits-RecognitionTask.zip>`_.

#     Args:
#         root (str|Path): Root directory of the UCF101 dataset.
#         annotation_path (str|Path): Path to the folder containing the split files
#         frames_per_clip (int): Number of frames in a clip
#         step_between_clips (int, optional): Number of frames between each clip
#         train (bool, optional): If ``True``, create a dataset from the train spit, oetherwise from the ``test`` split
#         transforms (callable, optional): A function/transform that takes in the video and annotation and returns the transformed versions
#     Returns:
#         (tuple): A 3-tuple with the following entries:
#             - video (Tensor): A set of video frames with shape (T,C, H, W) where T is the number of video frames
#             - audio (Tensor): A set of audio frames with shape (K,L) where `K` is the number of channels and `L` is the number of points
#             - label (int): Class of the video clip
#     """
#     def __init__(self, root: Union[str, Path], annotation_path: Union[str, Path], frames_per_clip:int, step_between_clips:int=1,
#                 train:bool=True, transforms:Optional[Callable]=None, num_workers:int=1, _video_width:int=0, _video_height:int=0,
#                 _video_min_dimension:int=0, _audio_samples:int=0)->None:
        
#         super().__init__(root=root, transforms=transforms)
        
#         extension=("avi",)
#         self.train=train
#         self.classes, class_to_idx=find_classes(self.root)
#         self.samples=make_dataset(self.root, class_to_idx, extension, is_valid_file=None)

# data_dirpath=Path('D:/data/UCF101')
# root=data_dirpath/'UCF-101'
# annotation_path=data_dirpath/'UCF101TrainTestSplits-RecognitionTask'
# dataset=UCF101(root=root, annotation_path=annotation_path, frames_per_clip=16, step_between_clips=2, train=True) 

In [34]:
# video_list=[x[0] for x in dataset.samples]
# print(f'{len(video_list)=}, {video_list[:2]=}')

len(video_list)=13320, video_list[:2]=['D:\\data\\UCF101\\UCF-101\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g01_c01.avi', 'D:\\data\\UCF101\\UCF-101\\ApplyEyeMakeup\\v_ApplyEyeMakeup_g01_c02.avi']


In [10]:

# file_path = "video_list.txt"
# content ='\n'.join(video_list)
# with open(file_path, "w") as file:
#     file.write(content)

In [11]:

data_dirpath=Path('D:/data/UCF101')
root=data_dirpath/'UCF-101'
video_fname=os.path.join(root, 'ApplyEyeMakeup','v_ApplyEyeMakeup_g01_c01.avi')
assert os.path.isfile(video_fname)

In [33]:
decoder=VideoDecoder(video_fname)
frame_rate=8
clip_duration=2
step_duration=1.5
clip_start_times, num_frames_per_clip, seconds_between_frames=compute_timestamp_clip_sampling_params(video_duration=decoder.metadata.duration_seconds, 
                                                                                                     average_fps=decoder.metadata.average_fps,
                                                                                                     frame_rate=frame_rate, 
                                                                                                     clip_duration=clip_duration, 
                                                                                                     step_duration=step_duration)
print(f"{clip_start_times=}, {num_frames_per_clip=}, {seconds_between_frames=}")

clip_start_times=tensor([0.0000, 1.5000, 3.0000, 4.5000]), num_frames_per_clip=16, seconds_between_frames=0.125


In [41]:
# clip start at 1.5
stop_time=max(0, video_duration-clip_duration)
clip_start_times=torch.arange(0, stop_time, step_duration)
clip_start_times

tensor([0.0000, 1.5000, 3.0000, 4.5000])

In [48]:
from torchcodec.samplers import clips_at_regular_timestamps, clips_at_random_timestamps

clips = clips_at_regular_timestamps(
    decoder,
    seconds_between_clip_starts=step_duration,
    num_frames_per_clip=num_frames_per_clip,
    seconds_between_frames=seconds_between_frames,
)
print(f"{clips.data.shape=}, {clips.data.dtype=}") # (N,T,C,H,W) where N is the number of clips, T is the number of frames per clips
print(f"{clips.pts_seconds.shape=}, {clips.pts_seconds.dtype=}") # (N,T) where N is the number of clips, T is the number of frames per clips
print(f"{clips.duration_seconds.shape=}, {clips.duration_seconds.dtype=}")  # (N,T)

encoder=VideoEncoder(frames=clips.data[1], frame_rate=frame_rate) # frame_rate is the frame rate of input video
encoded_frames=encoder.to_tensor(format='mp4')
print(clips.pts_seconds[1])
# play_video(encoded_frames)


clips.data.shape=torch.Size([4, 16, 3, 240, 320]), clips.data.dtype=torch.uint8
clips.pts_seconds.shape=torch.Size([4, 16]), clips.pts_seconds.dtype=torch.float64
clips.duration_seconds.shape=torch.Size([4, 16]), clips.duration_seconds.dtype=torch.float64
tensor([1.4800, 1.6000, 1.7200, 1.8400, 2.0000, 2.1200, 2.2400, 2.3600, 2.4800,
        2.6000, 2.7200, 2.8400, 3.0000, 3.1200, 3.2400, 3.3600],
       dtype=torch.float64)


In [49]:
clips = clips_at_regular_timestamps(
    decoder,
    seconds_between_clip_starts=step_duration,
    num_frames_per_clip=num_frames_per_clip,
    seconds_between_frames=seconds_between_frames,
    sampling_range_start=clip_start_times[1].item()
)
print(f"{clips.data.shape=}, {clips.data.dtype=}") # (N,T,C,H,W) where N is the number of clips, T is the number of frames per clips
print(f"{clips.pts_seconds.shape=}, {clips.pts_seconds.dtype=}") # (N,T) where N is the number of clips, T is the number of frames per clips
print(f"{clips.duration_seconds.shape=}, {clips.duration_seconds.dtype=}")  # (N,T)

encoder=VideoEncoder(frames=clips.data[0], frame_rate=frame_rate) # frame_rate is the frame rate of input video
encoded_frames=encoder.to_tensor(format='mp4')
print(clips.pts_seconds[0])
# play_video(encoded_frames)

clips.data.shape=torch.Size([3, 16, 3, 240, 320]), clips.data.dtype=torch.uint8
clips.pts_seconds.shape=torch.Size([3, 16]), clips.pts_seconds.dtype=torch.float64
clips.duration_seconds.shape=torch.Size([3, 16]), clips.duration_seconds.dtype=torch.float64
tensor([1.4800, 1.6000, 1.7200, 1.8400, 2.0000, 2.1200, 2.2400, 2.3600, 2.4800,
        2.6000, 2.7200, 2.8400, 3.0000, 3.1200, 3.2400, 3.3600],
       dtype=torch.float64)


In [37]:
clips = clips_at_random_timestamps(
    decoder,
    num_clips=1,
    num_frames_per_clip=num_frames_per_clip,
    seconds_between_frames=seconds_between_frames,
    policy="wrap",
)

print(f"{clips.data.shape=}, {clips.data.dtype=}") # (N,T,C,H,W) where N is the number of clips, T is the number of frames per clips
print(f"{clips.pts_seconds.shape=}, {clips.pts_seconds.dtype=}") # (N,T) where N is the number of clips, T is the number of frames per clips
print(f"{clips.duration_seconds.shape=}, {clips.duration_seconds.dtype=}")  # (N,T)

encoder=VideoEncoder(frames=clips.data[0], frame_rate=frame_rate) # frame_rate is the frame rate of input video
encoded_frames=encoder.to_tensor(format='mp4')
play_video(encoded_frames)

clips.data.shape=torch.Size([1, 16, 3, 240, 320]), clips.data.dtype=torch.uint8
clips.pts_seconds.shape=torch.Size([1, 16]), clips.pts_seconds.dtype=torch.float64
clips.duration_seconds.shape=torch.Size([1, 16]), clips.duration_seconds.dtype=torch.float64
